In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub


In [2]:
from langchain.agents import create_react_agent, AgentExecutor


In [ ]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """Extra weather data for a given city using the Weatherstack API."""
    url = (
        f"https//api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )
    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not retrieve weather data for {city}. Please check the city name and try again."

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%\n"
    )   
    

In [4]:
search_tool = TavilySearchResults(max_results=2)

In [5]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [8]:
prompt = hub.pull("hwchase17/react")

c:\Users\yuhan\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [ ]:
tools = [search_tool, get_weather_data] 

In [10]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [11]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [13]:
response = agent_executor.invoke({
    "input":(
        "Find the capital of China"
    )
})



> Entering new AgentExecutor chain...
I should use the search engine to find the answer
Action: tavily_search_results_json
Action Input: Capital of China[{'url': 'https://en.wikipedia.org/wiki/Beijing', 'content': 'Beijing, previously romanized as Peking, is the capital city of China. With more than 21.8 million residents, it is the world\'s most populous national capital city, as well as China\'s second-largest city by urban area, after Shanghai. It is located in Northern China and is governed as a provincial-level direct-administered municipality with 16 municipal districts. Beijing is mostly surrounded by Hebei Province and neighbors Tianjin Municipality to the southeast; together, the three divisions form the Jing-Jin-Ji cluster. [...] |  |  |  |\n --- \n| Preceded by Lin\'an (Song dynasty) | Capital of China (as Dadu of Yuan)  1264–1368 | Succeeded by Nanjing (Ming dynasty) |\n| Preceded by Nanjing (Ming dynasty) | Capital of China  1420–1928 | Succeeded by Nanjing (ROC) |\n| Pr